# RAG 第 6 课：Evaluation 入门

这一课我们正式进入 `RAG Evaluation`。

前面几课我们已经学了：
- 检索的基本流程
- chunking
- embeddings
- reranking
- hybrid search

但真实项目里，不能只靠“看起来结果还行”来判断系统好不好。
我们需要一些更明确的指标，来回答这些问题：

- 检索器有没有把正确文档找回来？
- 正确文档排得靠不靠前？
- 不同检索器之间，谁更稳？

这节课先聚焦 **检索阶段的 evaluation**，不急着讨论生成答案质量。

In [ ]:
# 先清理一下 GPU 缓存，避免前面课程占着显存
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 一个小型评测集

我们先准备一组小型文档和查询。
为了教学简单，我们直接手工指定每个 query 对应的 gold chunk，也就是“理想情况下应该命中的文档”。

这和真实 RAG 项目里的评测集思路是一样的，只不过真实项目里会更大、更复杂。

In [ ]:
import math
import re
from collections import Counter

import numpy as np

corpus = [
    'Error code E401 means authentication failed because the token is invalid or expired.',
    'Reset your password by opening Settings, then Security, then Reset Password.',
    'Chunking splits long documents into smaller passages for retrieval.',
    'Embeddings map text into vectors so semantic search becomes possible.',
    'Hybrid search combines lexical matching and dense retrieval scores.',
    'Reranking reorders retrieved candidates so the best evidence appears first.',
]

eval_queries = [
    {
        'query': 'What does E401 mean?',
        'gold_ids': [0],
    },
    {
        'query': 'How do I change my password?',
        'gold_ids': [1],
    },
    {
        'query': 'Why do we split documents into smaller chunks?',
        'gold_ids': [2],
    },
    {
        'query': 'What is the purpose of embeddings in retrieval?',
        'gold_ids': [3],
    },
    {
        'query': 'Why use reranking after retrieval?',
        'gold_ids': [5],
    },
]

for i, text in enumerate(corpus):
    print(f'chunk {i}: {text}')

## 2. 三个简单检索器

为了把评估逻辑讲清楚，我们继续沿用一个教学版设置：

- `lexical`：看词面匹配
- `dense`：看语义向量相似度
- `hybrid`：把前两者融合

这不是生产级实现，但足够帮助我们理解 evaluation 指标。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('ed') and len(token) > 4:
        token = token[:-2]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


vocab = sorted({token for text in corpus for token in tokenize(text)} | {token for item in eval_queries for token in tokenize(item['query'])})
rng = np.random.default_rng(7)
token_vectors = {token: rng.normal(0, 1, size=24) for token in vocab}


def lexical_score(query, text):
    q_tokens = tokenize(query)
    d_tokens = tokenize(text)
    d_set = set(d_tokens)

    score = 0.0
    for token in q_tokens:
        if token in d_set:
            score += 1.0
            if any(char.isdigit() for char in token):
                score += 1.5
    return score


def embed_text(text):
    tokens = tokenize(text)
    if not tokens:
        return np.zeros(24)
    vectors = [token_vectors[token] for token in tokens]
    return np.mean(vectors, axis=0)


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


doc_embeddings = [embed_text(text) for text in corpus]


def dense_score(query, text, text_embedding=None):
    q_emb = embed_text(query)
    t_emb = text_embedding if text_embedding is not None else embed_text(text)
    return cosine_similarity(q_emb, t_emb)


def min_max_normalize(scores):
    values = np.array(scores, dtype=float)
    low = values.min()
    high = values.max()
    if math.isclose(low, high):
        return [0.0 for _ in scores]
    return ((values - low) / (high - low)).tolist()


def rank_lexical(query):
    rows = []
    for idx, text in enumerate(corpus):
        rows.append({'doc_id': idx, 'score': lexical_score(query, text), 'text': text})
    return sorted(rows, key=lambda x: x['score'], reverse=True)


def rank_dense(query):
    rows = []
    for idx, text in enumerate(corpus):
        rows.append({'doc_id': idx, 'score': dense_score(query, text, doc_embeddings[idx]), 'text': text})
    return sorted(rows, key=lambda x: x['score'], reverse=True)


def rank_hybrid(query, alpha=0.6, beta=0.4):
    lexical_scores = [lexical_score(query, text) for text in corpus]
    dense_scores = [dense_score(query, text, doc_embeddings[idx]) for idx, text in enumerate(corpus)]

    lex_norm = min_max_normalize(lexical_scores)
    dense_norm = min_max_normalize(dense_scores)

    rows = []
    for idx, text in enumerate(corpus):
        score = alpha * lex_norm[idx] + beta * dense_norm[idx]
        rows.append({
            'doc_id': idx,
            'score': score,
            'lexical': lex_norm[idx],
            'dense': dense_norm[idx],
            'text': text,
        })
    return sorted(rows, key=lambda x: x['score'], reverse=True)

## 3. 先看一条 query 的排序结果

在上指标之前，先看一条 query 的完整排序会更直观。

In [ ]:
example_query = eval_queries[0]['query']
print('query:', example_query)
print('\nlexical top-3')
for row in rank_lexical(example_query)[:3]:
    print(f"doc={row['doc_id']} | score={row['score']:.3f}")
    print(row['text'])
    print()

print('dense top-3')
for row in rank_dense(example_query)[:3]:
    print(f"doc={row['doc_id']} | score={row['score']:.3f}")
    print(row['text'])
    print()

print('hybrid top-3')
for row in rank_hybrid(example_query)[:3]:
    print(f"doc={row['doc_id']} | hybrid={row['score']:.3f} | lexical={row['lexical']:.3f} | dense={row['dense']:.3f}")
    print(row['text'])
    print()

## 4. 检索指标到底在看什么

这一课我们先看 4 个最常见、也最容易上手的指标：

- `Hit@k`：前 k 个结果里，有没有至少命中一个 gold 文档
- `Precision@k`：前 k 个结果里，有多少比例是对的
- `Recall@k`：所有 gold 文档里，有多少已经被前 k 个结果召回
- `MRR`：第一个正确答案出现得有多靠前

其中 `MRR` 很重要，因为它特别关心“正确文档是否排得靠前”。
如果正确文档排在第 1 名，得分是 `1.0`；排第 2 名就是 `1/2=0.5`；排第 3 名就是 `1/3≈0.333`。

In [ ]:
def hit_at_k(ranked_ids, gold_ids, k):
    top_k = ranked_ids[:k]
    return 1.0 if any(doc_id in gold_ids for doc_id in top_k) else 0.0


def precision_at_k(ranked_ids, gold_ids, k):
    top_k = ranked_ids[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for doc_id in top_k if doc_id in gold_ids)
    return hits / k


def recall_at_k(ranked_ids, gold_ids, k):
    top_k = ranked_ids[:k]
    if not gold_ids:
        return 0.0
    hits = sum(1 for doc_id in top_k if doc_id in gold_ids)
    return hits / len(gold_ids)


def reciprocal_rank(ranked_ids, gold_ids):
    for idx, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in gold_ids:
            return 1.0 / idx
    return 0.0

## 5. 先手算一条 query

这一格的目标是让你真正把指标看懂，而不是只会调用函数。

In [ ]:
item = eval_queries[4]
query = item['query']
gold_ids = item['gold_ids']
ranked = rank_hybrid(query)
ranked_ids = [row['doc_id'] for row in ranked]

print('query:', query)
print('gold_ids:', gold_ids)
print('ranked_ids:', ranked_ids)
print('Hit@1   =', hit_at_k(ranked_ids, gold_ids, 1))
print('Hit@3   =', hit_at_k(ranked_ids, gold_ids, 3))
print('P@3     =', round(precision_at_k(ranked_ids, gold_ids, 3), 3))
print('Recall@3=', round(recall_at_k(ranked_ids, gold_ids, 3), 3))
print('RR      =', round(reciprocal_rank(ranked_ids, gold_ids), 3))

print('\nTop-3 documents:')
for row in ranked[:3]:
    print(f"doc={row['doc_id']} | score={row['score']:.3f}")
    print(row['text'])
    print()

## 6. 批量评估一个检索器

真实项目里不会只看一条 query，而是会在一整个评测集上取平均。

In [ ]:
def evaluate_retriever(eval_set, rank_fn, k=3):
    hit_scores = []
    precision_scores = []
    recall_scores = []
    rr_scores = []

    rows = []
    for item in eval_set:
        ranked = rank_fn(item['query'])
        ranked_ids = [row['doc_id'] for row in ranked]

        hit = hit_at_k(ranked_ids, item['gold_ids'], k)
        p_at_k = precision_at_k(ranked_ids, item['gold_ids'], k)
        r_at_k = recall_at_k(ranked_ids, item['gold_ids'], k)
        rr = reciprocal_rank(ranked_ids, item['gold_ids'])

        hit_scores.append(hit)
        precision_scores.append(p_at_k)
        recall_scores.append(r_at_k)
        rr_scores.append(rr)

        rows.append({
            'query': item['query'],
            'gold_ids': item['gold_ids'],
            'top_3': ranked_ids[:3],
            'hit@3': hit,
            'precision@3': p_at_k,
            'recall@3': r_at_k,
            'rr': rr,
        })

    summary = {
        'Hit@3': float(np.mean(hit_scores)),
        'Precision@3': float(np.mean(precision_scores)),
        'Recall@3': float(np.mean(recall_scores)),
        'MRR': float(np.mean(rr_scores)),
    }
    return summary, rows

In [ ]:
lexical_summary, lexical_rows = evaluate_retriever(eval_queries, rank_lexical, k=3)
dense_summary, dense_rows = evaluate_retriever(eval_queries, rank_dense, k=3)
hybrid_summary, hybrid_rows = evaluate_retriever(eval_queries, rank_hybrid, k=3)

print('lexical summary:', lexical_summary)
print('dense summary:  ', dense_summary)
print('hybrid summary: ', hybrid_summary)

## 7. 看每条 query 的细节

平均分有用，但还不够。
真实调系统时，你通常还会反过来看：到底是哪几条 query 拉低了分数。

In [ ]:
for row in hybrid_rows:
    print('query:', row['query'])
    print('gold_ids:', row['gold_ids'])
    print('top_3:', row['top_3'])
    print('hit@3:', row['hit@3'], '| precision@3:', round(row['precision@3'], 3), '| recall@3:', round(row['recall@3'], 3), '| rr:', round(row['rr'], 3))
    print('-' * 80)

## 8. 这些指标怎么理解

你可以先这样记：

- `Hit@k`：像不像一个合格的召回器
- `Precision@k`：前面塞给模型的内容是不是干净
- `Recall@k`：该找回来的东西有没有漏掉
- `MRR`：正确答案是不是排得足够靠前

在 RAG 里，`MRR` 和 `Hit@k` 往往都很常见，因为：
- 如果正确 chunk 根本没被召回，后面的生成再强也没用
- 如果正确 chunk 排得太靠后，也可能被后面的上下文截断掉

所以很多团队会同时看：
- `Recall@10` 或 `Hit@10`：看召回够不够
- `MRR` 或 `nDCG`：看排序质量够不够

## 9. 本课小结

这节课你要带走的核心是：

1. RAG 不能只靠主观感觉评估，要有检索指标。
2. `Hit@k / Precision@k / Recall@k / MRR` 是最基础的一组入门指标。
3. Evaluation 不只是看平均分，还要看失败样本。

下一课最自然的衔接就是：
**生成阶段的评估**，也就是当检索没问题之后，怎么判断答案质量。